# 13 · QKD security depth — finite keys, authentication, biased bases, live decoy

The earlier notebooks end with an *asymptotically* secure key over an *unauthenticated*
channel from an *idealized* single-photon source. This notebook demonstrates the four
upgrades that close those gaps, each runnable live on the two-process emulator:

1. **Finite-key security** — the honest secret length for a run of N pulses
   (Serfling-corrected QBER + failure-probability terms), not the N→∞ rate.
2. **Authenticated classical channel** — BB84's proof *requires* it; per-frame
   HMAC tags + anti-replay sequence numbers under a pre-shared key.
3. **Efficient (biased-basis) BB84** — sift ratio p²+(1−p)² > ½; key from Z–Z
   matches, phase error estimated from the fully-disclosed X–X matches.
4. **Decoy states on the live transport** — a weak-coherent Poisson source whose
   *measured* per-intensity gains and error rates feed the Lo–Ma–Chen bounds, closing
   the photon-number-splitting loophole with live statistics.

Run this on the SeQUeNCe-env kernel.

## 1 · Setup — two-process runner

In [ ]:
import sys, os, json, subprocess, time, math
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PKG = PROJECT_DIR / 'qne-sequence'
for p in (str(PROJECT_DIR), str(PKG)):
    if p not in sys.path:
        sys.path.insert(0, p)

_PORT = [57780]

def run_bb84(num_pulses=8000, fidelity=0.97, sample_fraction=0.2, seed=7, extra=()):
    """Launch Bob + Alice on loopback; return (alice_result, bob_result)."""
    _PORT[0] += 2
    port = _PORT[0]
    env = dict(os.environ)
    env['PYTHONPATH'] = f"{PKG}{os.pathsep}{PROJECT_DIR}"
    def spawn(role, peer):
        return subprocess.Popen(
            [sys.executable, '-m', 'qne_sequence.node_runner',
             '--role', role, '--name', role, '--peer', peer,
             '--host', '127.0.0.1', '--port', str(port),
             '--num-pulses', str(num_pulses), '--fidelity', str(fidelity),
             '--sample-fraction', str(sample_fraction), '--seed', str(seed), *extra],
            cwd=str(PKG), env=env, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    bob = spawn('bob', 'alice'); time.sleep(1); alice = spawn('alice', 'bob')
    a_out, a_err = alice.communicate(timeout=180)
    b_out, _ = bob.communicate(timeout=60)
    def parse(out, err=''):
        line = next((ln for ln in out.splitlines() if ln.startswith('{')), '')
        assert line, f"no result:\n{err}"
        return json.loads(line)
    return parse(a_out, a_err), parse(b_out)

a, b = run_bb84()
print(f"smoke: qber={a['qber']:.4f}  secure_key_bits={a['secure_key_bits']}  "
      f"keys match={a['key'] == b['key']}")

## 2 · Finite-key security

The asymptotic Shor–Preskill fraction trusts the sampled QBER as exact. For n key bits
and k sampled bits the honest length is

    ℓ = n·(1 − h(Q + μ)) − leak_EC − log₂(2/ε_cor) − 2·log₂(1/(2·ε_PA))

with the Serfling fluctuation μ ~ √(ln(1/ε)/k): a finite sample might have *missed*
errors, so the bound charges for the worst case. The curves show how much key the
finite-size terms cost — and why short noisy runs yield **zero**.

In [ ]:
from qne.finite_key import finite_key_length, planned_leak
from qne.bb84 import BB84Protocol

ns = np.logspace(3, 7, 25).astype(int)
fig, ax = plt.subplots(figsize=(7, 4.5))
for q in (0.01, 0.03, 0.05):
    rates = [finite_key_length(n, n // 10, q, planned_leak(n, q)).secret_bits / n
             for n in ns]
    line, = ax.semilogx(ns, rates, label=f'finite  Q={q:.0%}')
    asym = 1 - BB84Protocol.binary_entropy(q) - planned_leak(1, q)
    ax.axhline(asym, color=line.get_color(), ls=':', alpha=0.5)
ax.set_xlabel('key bits n (sample k = n/10)')
ax.set_ylabel('secret fraction ℓ/n')
ax.set_title('Finite-key rate vs block size (dotted = asymptotic)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# live: --finite-key sizes the Toeplitz output with the finite bound
a_fin, b_fin = run_bb84(num_pulses=12000, extra=('--finite-key',))
fk = a_fin['finite_key']
print(f"observed QBER {a_fin['qber']:.4f} -> Serfling-corrected upper bound {fk['qber_upper']:.4f}")
print(f"asymptotic accounting: {fk['asymptotic_bits']} bits")
print(f"finite-key secret:     {fk['secret_bits']} bits  "
      f"(= extracted length {a_fin['secure_key_bits']})")
print(f"both sides agree: {a_fin['finite_key'] == b_fin['finite_key']}, "
      f"keys match: {a_fin['key'] == b_fin['key']}")

## 3 · Authenticated classical channel

An attacker who can **rewrite** sifting traffic can man-in-the-middle all of BB84 — the
proof assumes authentication, which production QKD gets from Wegman–Carter MACs. The
emulator models the same wire protocol with HMAC-SHA256 + strictly-sequential frame
numbers under a pre-shared key: tampering, replay, or splicing kills the connection.

In [ ]:
from qne.auth import FrameAuthenticator, AuthError

def fresh():
    # one authenticated connection = one tx/rx counter pair; a frame that fails
    # verification kills the connection, so each attack demo gets a fresh one
    return FrameAuthenticator('psk'), FrameAuthenticator('psk')

tx, rx = fresh()
print('roundtrip:', rx.open(tx.seal(b'basis list')))

tx, rx = fresh()
frame = bytearray(tx.seal(b'qber result'))
frame[-1] ^= 1                                   # attacker flips one bit
try:
    rx.open(bytes(frame))
except AuthError as e:
    print('tampered frame  ->', e)

tx, rx = fresh()
ok = tx.seal(b'parities')
rx.open(ok)
try:
    rx.open(ok)                                  # attacker replays it
except AuthError as e:
    print('replayed frame  ->', e)

# live: every classical frame authenticated end-to-end
a_auth, b_auth = run_bb84(extra=('--auth-key', 'demo-psk'))
print(f"live run authenticated={a_auth['authenticated']}  "
      f"auth_failures={a_auth['auth_failures']}  keys match={a_auth['key'] == b_auth['key']}")

## 4 · Efficient BB84 — biased bases

Choosing Z with probability p > ½ on both sides lifts the sift ratio to p² + (1−p)²
(0.82 at p = 0.9, vs 0.5 unbiased). Security is kept honest by splitting the estimate:
the key comes from Z–Z matches, and **all** X–X matches are disclosed to bound the
phase error — rate 1 − h(e_z) − h(e_x).

In [ ]:
p = np.linspace(0.5, 0.99, 50)
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(p, p**2 + (1 - p)**2)
ax.axhline(0.5, color='r', lw=1, ls=':', label='unbiased BB84')
ax.set_xlabel('P(Z basis)'); ax.set_ylabel('sift ratio p² + (1−p)²')
ax.set_title('Efficient BB84: sifting beats the 50% coin flip')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

for bias in (0.5, 0.9):
    a_b, b_b = run_bb84(num_pulses=8000, extra=('--basis-bias', str(bias)))
    print(f"p(Z)={bias}: sift_ratio={b_b['sift_ratio'] and round(b_b['sift_ratio'], 3)}  "
          f"qber_z={a_b['qber']:.4f}  qber_x={a_b['qber_x']}  "
          f"secure_fraction={a_b['secure_fraction']:.3f}  "
          f"keys match={a_b['key'] == b_b['key']}")

## 5 · Decoy states on the live transport

A weak-coherent laser emits Poisson(μ) photons per pulse; multi-photon pulses leak to a
photon-number-splitting attacker. The decoy method sends three intensities and compares
their **measured** gains and error rates to lower-bound the single-photon yield Y₁
(Lo–Ma–Chen) and get a PNS-proof GLLP rate. Here the whole pipeline runs on live
traffic: Alice's source draws real photon numbers, fiber loss thins them per photon,
Bob's detector fires with 1 − (1−η)ⁿ, and the analysis sees only what was measured.

In [ ]:
DIST, ATT, EFF, MU_S, MU_D = 10.0, 0.2, 0.8, 0.6, 0.1
a_d, b_d = run_bb84(num_pulses=20000, fidelity=0.98,
                    extra=('--decoy', '--mu-signal', str(MU_S), '--mu-decoy', str(MU_D),
                           '--distance-km', str(DIST), '--attenuation', str(ATT),
                           '--efficiency', str(EFF)))
d = a_d['decoy']
eta = EFF * 10 ** (-(ATT * DIST) / 10.0)         # detector x fiber transmittance

labels = ('signal', 'decoy', 'vacuum')
analytic = {'signal': 1 - math.exp(-eta * MU_S),
            'decoy': 1 - math.exp(-eta * MU_D), 'vacuum': 0.0}
fig, ax = plt.subplots(figsize=(6.5, 4))
x = np.arange(3)
ax.bar(x - 0.18, [d['gains'][k] for k in labels], 0.36, label='measured gain')
ax.bar(x + 0.18, [analytic[k] for k in labels], 0.36, label='analytic 1−e^{−ημ}')
ax.set_xticks(x, labels); ax.set_yscale('log'); ax.set_ylim(1e-5, 1)
ax.set_ylabel('gain Q_μ'); ax.set_title('Live decoy gains vs weak-coherent model')
ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

print(f"E_signal={d['qber_signal']:.4f}  E_decoy={d['qber_decoy']:.4f}")
print(f"Lo–Ma–Chen: Y1_lower={d['Y1_lower']:.4f}  e1_upper={d['e1_upper']:.4f}")
print(f"GLLP secure key rate {d['secure_key_rate']:.5f}/pulse  "
      f"-> decoy budget {d['decoy_key_bits']} bits")
print(f"signal-pulse key reconciled + identical: {a_d['key'] == b_d['key']}")

## Notes

- The flags compose: `--auth-key` + `--finite-key` together give an authenticated,
  finite-key-sized run (decoy and biased bases are deliberately mutually exclusive).
- Everything here forwards to the slice via `deploy_fabric.run_sequence_bb84(...)`
  (same knob names) — the interesting WAN experiment is authentication cost vs RTT
  under `apply_classical_netem`.
- The repeater chain (notebook 12) accepts `--auth-key`/`--finite-key` too; finite-key
  over a noisy chain needs long blocks — at chain QBER ≈ 5% a 6k-bit block yields
  **zero** finite-key bits, which is the bound doing its job.